In [ ]:
import os
import sys
import subprocess
import shutil

def run_pip(args):
    command = [sys.executable, "-m", "pip", "install"] + args
    print("[bootstrap] Running:", " ".join(command))
    subprocess.check_call(command)

def detect_gpu_compute_caps():
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        print("[bootstrap] nvidia-smi not found; skipping GPU capability detection")
        return []

    result = subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=gpu_name,compute_cap",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("[bootstrap] GPU detection failed:")
        print(result.stderr.strip() or result.stdout.strip())
        return []

    entries = []
    for raw_line in result.stdout.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        name, _, cap_text = line.partition(",")
        name = name.strip()
        cap_text = cap_text.strip()
        try:
            capability = float(cap_text)
        except ValueError:
            print(f"[bootstrap] Could not parse compute capability from: {line}")
            continue
        entries.append((name, capability))
    return entries

gpu_entries = detect_gpu_compute_caps()
if gpu_entries:
    for gpu_name, capability in gpu_entries:
        print(f"[bootstrap] Detected GPU: {gpu_name} (compute capability {capability:.1f})")
else:
    print("[bootstrap] No GPU capability entries detected")

needs_p100_compatible_torch = any(capability < 7.0 for _, capability in gpu_entries)
if needs_p100_compatible_torch:
    print("[bootstrap] Detected pre-Volta GPU; installing PyTorch 2.4.1/cu124 for P100 compatibility")
    run_pip([
        "torch==2.4.1",
        "torchvision==0.19.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
    ])
else:
    print("[bootstrap] Torch downgrade not required")

print("[bootstrap] Installing notebook dependencies")
run_pip(["timm==0.9.16", "loguru", "omegaconf", "torchreid", "opencv-python-headless"])
print("[bootstrap] Environment setup complete")

In [ ]:
import os

script_content = r'''#!/usr/bin/env python3
"""09d Vehicle ReID ResNet101-IBN-a Training Script (subprocess-safe for P100)"""

import os
import sys
import time
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tv_models
from loguru import logger
from pathlib import Path
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Sampler
import shutil
import gc
import socket
import urllib.request
import traceback
import atexit

_LOG_PATH = Path("/kaggle/working/debug.log")
_LOG = open(_LOG_PATH, "w", buffering=1)

class TeeWriter:
    def __init__(self, orig, log):
        self.orig = orig
        self.log = log

    def write(self, s):
        self.orig.write(s)
        try:
            self.log.write(s)
        except Exception:
            pass

    def flush(self):
        self.orig.flush()
        try:
            self.log.flush()
        except Exception:
            pass

def main():
    sys.stdout = TeeWriter(sys.stdout, _LOG)
    sys.stderr = TeeWriter(sys.stderr, _LOG)

    def _shutdown():
        try:
            _LOG.close()
        except Exception:
            pass

    atexit.register(_shutdown)

    print("[DEBUG] Logging initialized")
    print(f"[DEBUG] Hostname: {os.uname().nodename if hasattr(os, 'uname') else 'unknown'}")

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
    print(f"PyTorch: {torch.__version__}")

    KAGGLE_INPUT_ROOT = Path("/kaggle/input")
    EXPECTED_WEIGHTS_DATASET_ROOT = KAGGLE_INPUT_ROOT / "mtmc-weights"
    EXPECTED_CITYFLOW_DATASET_ROOT = KAGGLE_INPUT_ROOT / "data-aicity-2023-track-2"
    CITYFLOW_CHECKPOINT = "/kaggle/input/mtmc-weights/reid/resnet101ibn_cityflowv2_384px_best.pth"
    VERI776_CHECKPOINT = "/kaggle/input/mtmc-weights/reid/resnet101ibn_veri776_best.pth"

    def dedupe_paths(paths):
        unique = []
        seen = set()
        for path in paths:
            text = str(path)
            if text in seen:
                continue
            seen.add(text)
            unique.append(path)
        return unique

    def find_named_paths(name, want_dir=None, max_results=50):
        if not KAGGLE_INPUT_ROOT.exists():
            return []
        matches = []
        for path in KAGGLE_INPUT_ROOT.rglob(name):
            if want_dir is True and not path.is_dir():
                continue
            if want_dir is False and not path.is_file():
                continue
            matches.append(path)
            if len(matches) >= max_results:
                break
        return sorted(matches)

    def resolve_checkpoint_path(expected_path):
        candidate = Path(expected_path)
        if candidate.exists():
            print(f"[DEBUG] Found checkpoint at expected path: {candidate}")
            return str(candidate)
        print(f"[DEBUG] Expected checkpoint missing: {candidate}")
        if not KAGGLE_INPUT_ROOT.exists():
            print(f"[DEBUG] Kaggle input root missing: {KAGGLE_INPUT_ROOT}")
            return None
        matches = find_named_paths(candidate.name, want_dir=False, max_results=20)
        if matches:
            print(f"[DEBUG] Resolved checkpoint {candidate.name}: {matches[0]}")
            if len(matches) > 1:
                preview = [str(match) for match in matches[:5]]
                print(f"[DEBUG] Additional checkpoint matches for {candidate.name}: {preview}")
            return str(matches[0])
        print(f"[DEBUG] No recursive match found for {candidate.name} under {KAGGLE_INPUT_ROOT}")
        return None

    def resolve_cityflow_train_root():
        candidate_bases = dedupe_paths(
            [EXPECTED_CITYFLOW_DATASET_ROOT]
            + find_named_paths("data-aicity-2023-track-2", want_dir=True, max_results=20)
        )
        candidate_train_roots = []
        for base in candidate_bases:
            for subpath in ("AIC22_Track1_MTMC_Tracking/train", "train"):
                candidate = base / subpath
                if candidate.exists():
                    candidate_train_roots.append(candidate)

        candidate_train_roots = dedupe_paths(candidate_train_roots)
        if candidate_train_roots:
            best_root = max(
                candidate_train_roots,
                key=lambda root: (len(list(root.glob("S*/c*"))), str(root)),
            )
            print(f"[DEBUG] Resolved CityFlowV2 train root from dataset directory: {best_root}")
            return best_root

        inferred_roots = []
        sample_pairs = {}
        for gt_path in find_named_paths("gt.txt", want_dir=False, max_results=200):
            if gt_path.parent.name != "gt":
                continue
            camera_dir = gt_path.parent.parent
            scene_dir = camera_dir.parent
            video_file = camera_dir / "vdo.avi"
            if not video_file.exists():
                continue
            if not camera_dir.name.startswith("c") or not scene_dir.name.startswith("S"):
                continue
            train_root = scene_dir.parent
            inferred_roots.append(train_root)
            sample_pairs.setdefault(str(train_root), (gt_path, video_file))

        inferred_roots = dedupe_paths(inferred_roots)
        if inferred_roots:
            scored_roots = []
            for root in inferred_roots:
                camera_count = len(list(root.glob("S*/c*")))
                scored_roots.append((camera_count, root))
            scored_roots.sort(key=lambda item: (-item[0], str(item[1])))
            best_root = scored_roots[0][1]
            sample_gt, sample_video = sample_pairs[str(best_root)]
            print(f"[DEBUG] Inferred CityFlowV2 train root via gt/video search: {best_root}")
            print(f"[DEBUG] Sample gt file: {sample_gt}")
            print(f"[DEBUG] Sample video file: {sample_video}")
            return best_root

        raise FileNotFoundError(
            "CityFlowV2 dataset not found anywhere under /kaggle/input. "
            "Expected dataset slug: data-aicity-2023-track-2"
        )

    if not EXPECTED_WEIGHTS_DATASET_ROOT.exists():
        print(
            f"[WARN] Expected weights dataset root missing: {EXPECTED_WEIGHTS_DATASET_ROOT}. "
            "Falling back to recursive search under /kaggle/input."
        )
    resolved_cityflow_checkpoint = resolve_checkpoint_path(CITYFLOW_CHECKPOINT)
    resolved_veri776_checkpoint = resolve_checkpoint_path(VERI776_CHECKPOINT)
    if resolved_cityflow_checkpoint is None:
        raise FileNotFoundError(
            f"Could not resolve {Path(CITYFLOW_CHECKPOINT).name} anywhere under {KAGGLE_INPUT_ROOT}"
        )
    CITYFLOW_CHECKPOINT = resolved_cityflow_checkpoint
    print(f"[DEBUG] Using CityFlowV2 checkpoint: {CITYFLOW_CHECKPOINT}")
    if resolved_veri776_checkpoint is not None:
        VERI776_CHECKPOINT = resolved_veri776_checkpoint
        print(f"[DEBUG] Using VeRi-776 checkpoint: {VERI776_CHECKPOINT}")
    else:
        print(
            f"[WARN] Could not resolve {Path(VERI776_CHECKPOINT).name} under {KAGGLE_INPUT_ROOT}. "
            "VeRi-776 fallback will be unavailable if the CityFlow checkpoint cannot be used."
        )

    CFG = {
        "dataset_root": "/kaggle/working/cityflowv2_reid",
        "weights_output": "/kaggle/working/resnet101ibn_cityflowv2_384px_best.pth",
        "checkpoint_dir": "/kaggle/working/checkpoints",
        "backbone": "resnet101_ibn_a",
        "feat_dim": 2048,
        "img_size": (384, 384),
        "gem_p": 3.0,
        "epochs": 120,
        "batch_size": 32,
        "gradient_accumulation_steps": 2,
        "eval_batch_size": 64,
        "num_instances": 4,
        "lr": 3e-4,
        "backbone_lr_factor": 0.1,
        "warmup_epochs": 5,
        "eta_min": 1e-6,
        "weight_decay": 5e-4,
        "label_smoothing": 0.05,
        "triplet_margin": 0.3,
        "circle_m": 0.25,
        "circle_gamma": 80,
        "triplet_weight": 1.0,
        "circle_weight": 0.0,
        "id_weight": 1.0,
        "random_erasing_prob": 0.6,
        "color_jitter": True,
        "eval_every": 5,
        "fp16": True,
    }

    os.makedirs(CFG["checkpoint_dir"], exist_ok=True)
    print(json.dumps(CFG, indent=2, default=str))
    print(
        f"Physical batch size: {CFG['batch_size']} | "
        f"gradient accumulation: {CFG['gradient_accumulation_steps']} | "
        f"effective batch size: {CFG['batch_size'] * CFG['gradient_accumulation_steps']}"
    )

    RESUME_FROM = CITYFLOW_CHECKPOINT
    RESUME_EPOCH = 0  # Reset epoch counter; this run fine-tunes from pretrained weights
    print({"resume_from": RESUME_FROM, "resume_epoch": RESUME_EPOCH})

    # Build ReID crops from raw CityFlowV2 videos

    RAW_ROOT = resolve_cityflow_train_root()
    print(f"[DEBUG] Using CityFlowV2 raw train root: {RAW_ROOT}")

    CROP_ROOT = Path("/kaggle/working/cityflowv2_crops")
    MAX_SAMPLES_PER_TRACK = 15
    MIN_BOX_AREA = 2000
    MIN_BOX_SIDE = 30


    if CROP_ROOT.exists():
        for existing_file in CROP_ROOT.glob("*.jpg"):
            existing_file.unlink()
    else:
        CROP_ROOT.mkdir(parents=True, exist_ok=True)

    camera_dirs = sorted(path for path in RAW_ROOT.glob("S*/c*") if path.is_dir())
    print(f"Found {len(camera_dirs)} cameras under {RAW_ROOT}")

    all_crops = {}
    total_crop_count = 0

    for cam_dir in camera_dirs:
        scene = cam_dir.parent.name
        cam = cam_dir.name
        cam_id = f"{scene}_{cam}"
        gt_file = cam_dir / "gt" / "gt.txt"
        video_file = cam_dir / "vdo.avi"

        if not gt_file.exists() or not video_file.exists():
            print(f"Skip {cam_id}: missing gt or video")
            continue

        detections = defaultdict(list)
        with gt_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                frame_id = int(parts[0])
                track_id = int(parts[1])
                x = int(float(parts[2]))
                y = int(float(parts[3]))
                w = int(float(parts[4]))
                h = int(float(parts[5]))
                if w * h < MIN_BOX_AREA or w < MIN_BOX_SIDE or h < MIN_BOX_SIDE:
                    continue
                detections[track_id].append((frame_id, x, y, w, h))

        samples = {}
        for track_id, dets in detections.items():
            dets.sort(key=lambda item: item[0])
            if len(dets) <= MAX_SAMPLES_PER_TRACK:
                sampled = dets
            else:
                sampled_indices = np.linspace(0, len(dets) - 1, num=MAX_SAMPLES_PER_TRACK, dtype=int)
                sampled = [dets[index] for index in sampled_indices]
            samples[track_id] = sampled

        dets_by_frame = defaultdict(list)
        for track_id, sampled_dets in samples.items():
            for frame_id, x, y, w, h in sampled_dets:
                dets_by_frame[frame_id].append((track_id, x, y, w, h))

        cap = cv2.VideoCapture(str(video_file))
        if not cap.isOpened():
            print(f"Skip {cam_id}: failed to open video")
            continue

        camera_crop_count = 0
        for frame_id in sorted(dets_by_frame.keys()):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
            ok, frame = cap.read()
            if not ok:
                continue
            frame_height, frame_width = frame.shape[:2]
            for track_id, x, y, w, h in dets_by_frame[frame_id]:
                x1 = max(0, x)
                y1 = max(0, y)
                x2 = min(frame_width, x + w)
                y2 = min(frame_height, y + h)
                if x2 <= x1 or y2 <= y1:
                    continue
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                file_name = f"{track_id:04d}_{scene}_{cam}_f{frame_id:06d}.jpg"
                crop_path = CROP_ROOT / file_name
                if not cv2.imwrite(str(crop_path), crop):
                    continue
                all_crops.setdefault(track_id, {}).setdefault(cam_id, []).append(
                    (str(crop_path), frame_id)
                )
                camera_crop_count += 1
        cap.release()

        total_crop_count += camera_crop_count
        print(f"  {cam_id}: {len(detections)} vehicles, {camera_crop_count} crops")
        gc.collect()

    print(f"\nTotal vehicles: {len(all_crops)}")
    print(f"Total crops: {total_crop_count}")

    # Create train/query/gallery splits from extracted crops

    REID_ROOT = Path(CFG["dataset_root"])
    if REID_ROOT.exists():
        shutil.rmtree(REID_ROOT)
    for split_name in ["train", "query", "gallery"]:
        (REID_ROOT / split_name).mkdir(parents=True, exist_ok=True)

    multi_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) >= 2]
    single_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) == 1]

    rng = np.random.default_rng(42)
    rng.shuffle(multi_cam_ids)

    n_train = int(len(multi_cam_ids) * 0.7)
    train_ids = set(multi_cam_ids[:n_train])
    eval_ids = set(multi_cam_ids[n_train:])

    counts = {"train": 0, "query": 0, "gallery": 0}

    for track_id in train_ids:
        for cam_id, items in all_crops[track_id].items():
            for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
                destination = REID_ROOT / "train" / Path(crop_path).name
                shutil.copy2(crop_path, destination)
                counts["train"] += 1

    for track_id in eval_ids:
        for cam_id, items in all_crops[track_id].items():
            ordered_items = sorted(items, key=lambda item: item[1])
            if not ordered_items:
                continue
            query_source, _ = ordered_items[0]
            shutil.copy2(query_source, REID_ROOT / "query" / Path(query_source).name)
            counts["query"] += 1
            for gallery_source, _ in ordered_items[1:]:
                shutil.copy2(gallery_source, REID_ROOT / "gallery" / Path(gallery_source).name)
                counts["gallery"] += 1

    for track_id in single_cam_ids:
        for cam_id, items in all_crops[track_id].items():
            for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
                destination = REID_ROOT / "gallery" / Path(crop_path).name
                shutil.copy2(crop_path, destination)
                counts["gallery"] += 1

    print(f"Splits: train={counts['train']}, query={counts['query']}, gallery={counts['gallery']}")
    print(f"Train IDs: {len(train_ids)}, Eval IDs: {len(eval_ids)}, Distractor IDs: {len(single_cam_ids)}")

    def parse_cityflowv2(root: str):
        train, query, gallery = [], [], []

        for split_name, split_list in [("train", train), ("query", query), ("gallery", gallery)]:
            split_dir = os.path.join(root, split_name)
            if not os.path.isdir(split_dir):
                raise FileNotFoundError(f"CityFlowV2 ReID split not found: {split_dir}")
            for fname in sorted(os.listdir(split_dir)):
                if not fname.endswith(".jpg"):
                    continue
                parts = fname.split("_")
                if len(parts) < 4:
                    continue
                pid = int(parts[0])
                cam_name = parts[1] + "_" + parts[2]
                split_list.append((os.path.join(split_dir, fname), pid, cam_name))

        all_cams = sorted({cam for _, _, cam in train + query + gallery})
        cam2id = {cam: idx for idx, cam in enumerate(all_cams)}
        train = [(path, pid, cam2id[cam]) for path, pid, cam in train]
        query = [(path, pid, cam2id[cam]) for path, pid, cam in query]
        gallery = [(path, pid, cam2id[cam]) for path, pid, cam in gallery]

        train_pids = sorted(set(pid for _, pid, _ in train))
        pid2label = {pid: label for label, pid in enumerate(train_pids)}
        train = [(path, pid2label[pid], cam) for path, pid, cam in train]

        return train, query, gallery, len(train_pids), len(all_cams)


    train_data, query_data, gallery_data, NUM_CLASSES, NUM_CAMERAS = parse_cityflowv2(CFG["dataset_root"])
    print(f"Train: {len(train_data)} images, {NUM_CLASSES} IDs, {NUM_CAMERAS} cameras")
    print(f"Query: {len(query_data)}, Gallery: {len(gallery_data)}")


    class ReIDDataset(Dataset):
        def __init__(self, data, transform=None):
            self.data = data
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            path, pid, cam = self.data[idx]
            image = Image.open(path).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            return image, pid, cam, path


    class PKSampler(Sampler):
        def __init__(self, data, p=12, k=4):
            self.p = p
            self.k = k
            self.pid_to_indices = defaultdict(list)
            for idx, (_, pid, _) in enumerate(data):
                self.pid_to_indices[pid].append(idx)
            self.pids = list(self.pid_to_indices.keys())
            self.batch_size = p * k

        def __iter__(self):
            pids = list(self.pids)
            np.random.shuffle(pids)
            batch = []
            for pid in pids:
                indices = self.pid_to_indices[pid]
                replace = len(indices) < self.k
                selected = np.random.choice(indices, size=self.k, replace=replace).tolist()
                batch.extend(selected)
                if len(batch) >= self.batch_size:
                    yield from batch[: self.batch_size]
                    batch = batch[self.batch_size :]
            if batch:
                yield from batch

        def __len__(self):
            return len(self.pids) * self.k


    height, width = CFG["img_size"]
    train_transform = T.Compose([
        T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.Pad(10),
        T.RandomCrop((height, width)),
        T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.05),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=CFG["random_erasing_prob"], value="random"),
    ])
    test_transform = T.Compose([
        T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    p = CFG["batch_size"] // CFG["num_instances"]
    k = CFG["num_instances"]
    assert p == 8 and k == 4, f"Expected PK sampling 8x4, got {p}x{k}"
    train_loader = DataLoader(
        ReIDDataset(train_data, train_transform),
        batch_size=CFG["batch_size"],
        sampler=PKSampler(train_data, p=p, k=k),
        num_workers=4,
        pin_memory=True,
        drop_last=True,
    )
    query_loader = DataLoader(
        ReIDDataset(query_data, test_transform),
        batch_size=CFG.get("eval_batch_size", 64),
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
    gallery_loader = DataLoader(
        ReIDDataset(gallery_data, test_transform),
        batch_size=CFG.get("eval_batch_size", 64),
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    print(f"Train loader: {len(train_loader)} batches, batch={CFG['batch_size']}")
    print(
        f"PK sampling: p={p}, k={k} | gradient accumulation={CFG['gradient_accumulation_steps']} | "
        f"effective batch={CFG['batch_size'] * CFG['gradient_accumulation_steps']}"
    )


    IBN_NET_RESNET101_A_URL = "https://github.com/XingangPan/IBN-Net/releases/download/v1.0/resnet101_ibn_a-59ea0ac6.pth"
    IBN_NET_RESNET101_A_LOCAL_PATH = "/kaggle/working/resnet101_ibn_a.pth"
    STEP6_MODEL_ERROR_PATH = "/kaggle/working/STEP6_MODEL_ERROR.txt"


    class IBN_a(nn.Module):
        def __init__(self, planes):
            super().__init__()
            half = planes // 2
            self.IN = nn.InstanceNorm2d(half, affine=True)
            self.BN = nn.BatchNorm2d(planes - half)

        def forward(self, x):
            split = x.shape[1] // 2
            return torch.cat([self.IN(x[:, :split]), self.BN(x[:, split:])], dim=1)


    class GeM(nn.Module):
        def __init__(self, p=3.0, eps=1e-6):
            super().__init__()
            self.p = nn.Parameter(torch.ones(1) * p)
            self.eps = eps

        def forward(self, x):
            return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


    class ResNet101IBN(nn.Module):
        def __init__(self, num_classes, feat_dim=2048, last_stride=1, gem_p=3.0, pretrained=True):
            super().__init__()
            base = tv_models.resnet101(weights=None)
            for layer in [base.layer1, base.layer2, base.layer3]:
                for block in layer:
                    if hasattr(block, "bn1"):
                        block.bn1 = IBN_a(block.bn1.num_features)
            load_result = None
            if pretrained:
                state_dict = None
                if os.path.exists(IBN_NET_RESNET101_A_LOCAL_PATH):
                    print(f"Using cached pretrained weights: {IBN_NET_RESNET101_A_LOCAL_PATH}")
                else:
                    try:
                        print(f"Downloading pretrained weights from {IBN_NET_RESNET101_A_URL}")
                        print(f"Saving pretrained weights to {IBN_NET_RESNET101_A_LOCAL_PATH}")
                        original_timeout = socket.getdefaulttimeout()
                        socket.setdefaulttimeout(120)
                        try:
                            urllib.request.urlretrieve(
                                IBN_NET_RESNET101_A_URL,
                                IBN_NET_RESNET101_A_LOCAL_PATH,
                            )
                        finally:
                            socket.setdefaulttimeout(original_timeout)
                        print(f"Pretrained weights downloaded to {IBN_NET_RESNET101_A_LOCAL_PATH}")
                    except Exception as download_error:
                        print(f"Direct download failed: {download_error}")
                        print("Falling back to torch.hub.load_state_dict_from_url")
                        state_dict = torch.hub.load_state_dict_from_url(
                            IBN_NET_RESNET101_A_URL,
                            map_location="cpu",
                            progress=True,
                        )
                if state_dict is None:
                    print(f"Loading pretrained weights from {IBN_NET_RESNET101_A_LOCAL_PATH}")
                    state_dict = torch.load(IBN_NET_RESNET101_A_LOCAL_PATH, map_location="cpu")
                load_result = base.load_state_dict(state_dict, strict=False)
                print(f"Missing keys: {load_result.missing_keys}")
                print(f"Unexpected keys: {load_result.unexpected_keys}")
            if load_result is not None:
                assert set(load_result.missing_keys) <= {"fc.weight", "fc.bias"}, f"Unexpected missing keys: {load_result.missing_keys}"
                allowed_unexpected_keys = {"fc.weight", "fc.bias", "classifier.weight", "classifier.bias"}
                assert set(load_result.unexpected_keys) <= allowed_unexpected_keys, f"Unexpected extra keys: {load_result.unexpected_keys}"
            if last_stride == 1:
                for module in base.layer4.modules():
                    if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                        module.stride = (1, 1)
            self.conv1 = base.conv1
            self.bn1 = base.bn1
            self.relu = base.relu
            self.maxpool = base.maxpool
            self.layer1 = base.layer1
            self.layer2 = base.layer2
            self.layer3 = base.layer3
            self.layer4 = base.layer4
            self.pool = GeM(p=gem_p)
            self.feat_dim = feat_dim
            self.bottleneck = nn.BatchNorm1d(feat_dim)
            self.bottleneck.bias.requires_grad_(False)
            nn.init.constant_(self.bottleneck.weight, 1)
            nn.init.constant_(self.bottleneck.bias, 0)
            self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
            nn.init.normal_(self.classifier.weight, std=0.001)

        def forward(self, x):
            x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)
            global_feat = self.pool(x).view(x.size(0), -1)
            bn_feat = self.bottleneck(global_feat)
            if self.training:
                return self.classifier(bn_feat), global_feat, bn_feat
            return F.normalize(bn_feat, p=2, dim=1)


    try:
        model_kwargs = dict(
            num_classes=NUM_CLASSES,
            feat_dim=CFG["feat_dim"],
            last_stride=1,
            gem_p=CFG["gem_p"],
        )
        model = ResNet101IBN(pretrained=False, **model_kwargs)
        if os.path.exists(CITYFLOW_CHECKPOINT):
            print(f"Loading CityFlowV2 fine-tuned checkpoint from {CITYFLOW_CHECKPOINT}")
            ckpt = torch.load(CITYFLOW_CHECKPOINT, map_location=DEVICE, weights_only=False)
            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                model_state = ckpt["model_state_dict"]
            elif isinstance(ckpt, dict) and "state_dict" in ckpt:
                model_state = ckpt["state_dict"]
            else:
                model_state = ckpt
            if any(key.startswith("module.") for key in model_state):
                model_state = {key[len("module."):] if key.startswith("module.") else key: value for key, value in model_state.items()}
            load_result = model.load_state_dict(model_state, strict=False)
            print(f"Loaded checkpoint: missing={load_result.missing_keys}, unexpected={load_result.unexpected_keys}")
        elif os.path.exists(VERI776_CHECKPOINT):
            print(f"Loading VeRi-776 pretrained weights from {VERI776_CHECKPOINT}")
            veri_state = torch.load(VERI776_CHECKPOINT, map_location="cpu", weights_only=False)
            if isinstance(veri_state, dict) and "state_dict" in veri_state:
                veri_state = veri_state["state_dict"]
            if any(key.startswith("module.") for key in veri_state):
                veri_state = {key[len("module."):] if key.startswith("module.") else key: value for key, value in veri_state.items()}
            model_state = model.state_dict()
            loaded, skipped = [], []
            for key, value in veri_state.items():
                if key in model_state and hasattr(value, "shape") and value.shape == model_state[key].shape:
                    model_state[key] = value
                    loaded.append(key)
                else:
                    skipped.append(key)
            load_result = model.load_state_dict(model_state, strict=False)
            print(f"Loaded {len(loaded)} VeRi-776 keys; skipped {len(skipped)}")
            print(f"VeRi-776 load_state_dict: missing={load_result.missing_keys}, unexpected={load_result.unexpected_keys}")
            if skipped:
                print(f"Skipped keys sample: {skipped[:10]}")
        else:
            print(f"Pretrained checkpoints not found at {CITYFLOW_CHECKPOINT} or {VERI776_CHECKPOINT}; using ImageNet initialization")
            model = ResNet101IBN(pretrained=True, **model_kwargs)
        model = model.to(DEVICE)
        if torch.cuda.device_count() > 1:
            print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
            model = nn.DataParallel(model)

        def unwrap_model(model):
            return model.module if hasattr(model, "module") else model

        def get_model_state_dict(model):
            return unwrap_model(model).state_dict()

        def load_model_state_dict(model, state_dict, strict=True):
            if any(key.startswith("module.") for key in state_dict):
                state_dict = {key[len("module."):]: value for key, value in state_dict.items()}
            return unwrap_model(model).load_state_dict(state_dict, strict=strict)

        num_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"ResNet101-IBN-a params: {num_params:.1f}M")
    except Exception:
        error_text = traceback.format_exc()
        with open(STEP6_MODEL_ERROR_PATH, "w", encoding="utf-8") as handle:
            handle.write(error_text)
        print(f"STEP6 model creation failed. Full traceback written to {STEP6_MODEL_ERROR_PATH}")
        raise

    class CrossEntropyLabelSmooth(nn.Module):
        def __init__(self, num_classes, epsilon=0.1):
            super().__init__()
            self.num_classes = num_classes
            self.epsilon = epsilon
            self.logsoftmax = nn.LogSoftmax(dim=1)

        def forward(self, inputs, targets):
            log_probs = self.logsoftmax(inputs)
            targets_oh = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            targets_smooth = (1 - self.epsilon) * targets_oh + self.epsilon / self.num_classes
            return (-targets_smooth * log_probs).mean(0).sum()


    class TripletLoss(nn.Module):
        def __init__(self, margin=0.3):
            super().__init__()
            self.ranking_loss = nn.MarginRankingLoss(margin=margin)

        def forward(self, inputs, targets):
            n = inputs.size(0)
            dist = torch.cdist(inputs, inputs, p=2)
            mask = targets.unsqueeze(0).eq(targets.unsqueeze(1))
            dist_ap = torch.stack([dist[i][mask[i]].max() for i in range(n)])
            dist_an = torch.stack([dist[i][~mask[i]].min() for i in range(n)])
            return self.ranking_loss(dist_an, dist_ap, torch.ones_like(dist_an))


    class CircleLoss(nn.Module):
        def __init__(self, m=0.25, gamma=80):
            super().__init__()
            self.m = m
            self.gamma = gamma
            self.soft_plus = nn.Softplus()

        def forward(self, inputs, targets):
            inputs = F.normalize(inputs, p=2, dim=1)
            n = inputs.size(0)
            sim = inputs @ inputs.t()
            mask = targets.unsqueeze(0).eq(targets.unsqueeze(1))
            op, on = 1 + self.m, -self.m
            delta_p, delta_n = 1 - self.m, self.m
            loss = 0.0
            for i in range(n):
                pos_sim = sim[i][mask[i]]
                neg_sim = sim[i][~mask[i]]
                ap = torch.clamp_min(-pos_sim.detach() + op, 0)
                an = torch.clamp_min(neg_sim.detach() + on, 0)
                logit_p = -ap * (pos_sim - delta_p) * self.gamma
                logit_n = an * (neg_sim - delta_n) * self.gamma
                loss = loss + self.soft_plus(torch.logsumexp(logit_n, 0) + torch.logsumexp(-logit_p, 0))
            return loss / n


    id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=CFG["label_smoothing"])
    triplet_loss_fn = TripletLoss(margin=CFG["triplet_margin"])
    circle_loss_fn = CircleLoss(m=CFG["circle_m"], gamma=CFG["circle_gamma"])
    print("Losses configured: ID + triplet + circle")

    model_for_optim = unwrap_model(model)

    params = [
        {
            "params": list(model_for_optim.conv1.parameters()) + list(model_for_optim.bn1.parameters()) + list(model_for_optim.layer1.parameters()) + list(model_for_optim.layer2.parameters()) + list(model_for_optim.layer3.parameters()) + list(model_for_optim.layer4.parameters()),
            "lr": CFG["lr"] * CFG["backbone_lr_factor"],
        },
        {"params": model_for_optim.pool.parameters(), "lr": CFG["lr"]},
        {"params": model_for_optim.bottleneck.parameters(), "lr": CFG["lr"]},
        {"params": model_for_optim.classifier.parameters(), "lr": CFG["lr"]},
    ]
    optimizer = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=CFG["warmup_epochs"],
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG["epochs"] - CFG["warmup_epochs"],
        eta_min=CFG["eta_min"],
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[CFG["warmup_epochs"]],
    )
    scaler = torch.amp.GradScaler("cuda") if CFG["fp16"] and torch.cuda.is_available() else None
    print(
        f"Optimizer ready. backbone_lr={CFG['lr'] * CFG['backbone_lr_factor']}, "
        f"head_lr={CFG['lr']}, scheduler=cosine, warmup={CFG['warmup_epochs']}"
    )

    @torch.no_grad()
    def extract_features(model, loader, device):
        model.eval()
        feats, pids, cams = [], [], []
        for imgs, pid, cam, _ in loader:
            imgs = imgs.to(device)
            feats_main = model(imgs)
            feats_flip = model(torch.flip(imgs, dims=[3]))
            feats_avg = F.normalize((feats_main + feats_flip) / 2, p=2, dim=1)
            feats.append(feats_avg.cpu().numpy())
            pids.append(pid.numpy())
            cams.append(cam.numpy())
        return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)


    def eval_reid(qf, qp, query_cams, gf, gp, gallery_cams, max_rank=50):
        dist = 1 - qf @ gf.T
        all_ap = []
        all_cmc = []
        for i in range(dist.shape[0]):
            order = np.argsort(dist[i])
            valid = ~((gp[order] == qp[i]) & (gallery_cams[order] == query_cams[i]))
            matches = (gp[order][valid] == qp[i]).astype(np.int32)
            if matches.sum() == 0:
                continue
            cmc = matches.cumsum()
            cmc = (cmc > 0).astype(np.float32)
            cmc_vec = np.zeros(max_rank, dtype=np.float32)
            upto = min(max_rank, len(cmc))
            cmc_vec[:upto] = cmc[:upto]
            all_cmc.append(cmc_vec)
            cum_tp = matches.cumsum()
            precision = cum_tp / (np.arange(len(matches)) + 1)
            all_ap.append((precision * matches).sum() / matches.sum())
        mAP = float(np.mean(all_ap)) if all_ap else 0.0
        cmc = np.mean(all_cmc, axis=0) if all_cmc else np.zeros(max_rank, dtype=np.float32)
        return mAP, cmc

    best_mAP = 0.0
    history = []

    if RESUME_FROM and os.path.exists(RESUME_FROM):
        print(f"Loading fine-tune checkpoint from {RESUME_FROM}")
        ckpt = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif isinstance(ckpt, dict) and "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        else:
            state_dict = ckpt
        load_result = load_model_state_dict(model, state_dict, strict=False)
        print(f"Fine-tune checkpoint loaded: missing={load_result.missing_keys}, unexpected={load_result.unexpected_keys}")
        RESUME_EPOCH = 0
        print(f"Starting fine-tuning from epoch {RESUME_EPOCH}")
        if isinstance(ckpt, dict) and "mAP" in ckpt:
            best_mAP = float(ckpt["mAP"])
            print(f"Previous best mAP: {best_mAP:.4f}")

    for epoch in range(RESUME_EPOCH, CFG["epochs"]):
        model.train()
        running = {"loss": 0.0, "id": 0.0, "tri": 0.0, "cir": 0.0, "n": 0}
        start_time = time.time()
        gradient_accumulation_steps = CFG.get("gradient_accumulation_steps", 1)
        optimizer.zero_grad(set_to_none=True)

        for step, (imgs, pids, _, _) in enumerate(train_loader):
            imgs = imgs.to(DEVICE)
            pids = pids.to(DEVICE).long()

            with torch.amp.autocast("cuda", enabled=scaler is not None):
                cls_score, global_feat, bn_feat = model(imgs)
                loss_id = id_loss_fn(cls_score, pids) * CFG["id_weight"]
                loss_tri = triplet_loss_fn(global_feat, pids) * CFG["triplet_weight"]
                if CFG["circle_weight"] > 0:
                    loss_cir = circle_loss_fn(global_feat, pids) * CFG["circle_weight"]
                else:
                    loss_cir = torch.tensor(0.0, device=DEVICE)
                loss = loss_id + loss_tri + loss_cir

            loss_value = loss.item()
            loss = loss / gradient_accumulation_steps
            should_step = (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader)

            if scaler is not None:
                scaler.scale(loss).backward()
                if should_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            running["loss"] += loss_value
            running["id"] += loss_id.item()
            running["tri"] += loss_tri.item()
            running["cir"] += loss_cir.item()
            running["n"] += 1

        scheduler.step()
        n_batches = max(running["n"], 1)
        elapsed = time.time() - start_time
        lr = optimizer.param_groups[0]["lr"]
        print(
            f"E{epoch:02d} | {elapsed:.0f}s | LR={lr:.6f} | "
            f"L={running['loss']/n_batches:.4f} ID={running['id']/n_batches:.4f} "
            f"Tri={running['tri']/n_batches:.4f} Cir={running['cir']/n_batches:.4f}"
        )

        if (epoch + 1) % CFG["eval_every"] == 0 or epoch == CFG["epochs"] - 1:
            qf, qp, qc = extract_features(model, query_loader, DEVICE)
            gf, gp, gallery_cams = extract_features(model, gallery_loader, DEVICE)
            mAP, cmc = eval_reid(qf, qp, qc, gf, gp, gallery_cams)
            print(f"  >> mAP={mAP:.4f}, R1={cmc[0]:.4f}, R5={cmc[4]:.4f}")
            history.append({"epoch": epoch, "mAP": float(mAP), "R1": float(cmc[0])})

            state_dict = get_model_state_dict(model)

            torch.save(
                {
                    "state_dict": state_dict,
                    "epoch": epoch,
                    "optimizer": optimizer.state_dict(),
                    "mAP": float(mAP),
                },
                f"{CFG['checkpoint_dir']}/ckpt_e{epoch:02d}.pth",
            )

            if mAP > best_mAP:
                best_mAP = mAP
                torch.save({"state_dict": state_dict, "epoch": epoch, "mAP": float(mAP)}, CFG["weights_output"])
                print(f"  * New best mAP: {best_mAP:.4f}")

    print(f"Training complete. Best mAP: {best_mAP:.4f}")


    if os.path.exists(CFG["weights_output"]):
        ckpt = torch.load(CFG["weights_output"], map_location="cpu", weights_only=False)
        load_model_state_dict(model, ckpt["state_dict"])
        model.to(DEVICE)

        qf, qp, qc = extract_features(model, query_loader, DEVICE)
        gf, gp, gallery_cams = extract_features(model, gallery_loader, DEVICE)
        mAP, cmc = eval_reid(qf, qp, qc, gf, gp, gallery_cams)
        print(f"Final: mAP={mAP:.4f}, R1={cmc[0]:.4f}, R5={cmc[4]:.4f}, R10={cmc[9]:.4f}")
        print(json.dumps(history, indent=2))
    else:
        print("No best weights saved - model may not have been evaluated or mAP was 0")

    output_name = "resnet101ibn_cityflowv2_384px_best.pth"
    src = CFG["weights_output"]
    dst = f"/kaggle/working/{output_name}"
    if os.path.exists(CFG["weights_output"]):
        if os.path.abspath(src) != os.path.abspath(dst):
            shutil.copy2(src, dst)
        print(f"Weights saved: {dst}")
        print(f"Size: {os.path.getsize(src if os.path.abspath(src) == os.path.abspath(dst) else dst) / 1e6:.1f} MB")
    else:
        print("No best weights to copy")

    with open("/kaggle/working/training_history_resnet101ibn.json", "w", encoding="utf-8") as handle:
        json.dump({"config": CFG, "history": history, "best_mAP": float(best_mAP)}, handle, indent=2, default=str)

    ckpt = torch.load("/kaggle/working/resnet101ibn_cityflowv2_384px_best.pth", map_location="cpu", weights_only=False)
    keys = list(ckpt["state_dict"].keys())
    print(f"State dict keys: {len(keys)}")
    print(f"Key samples: {keys[:5]} ... {keys[-5:]}")
    assert any(key.startswith("conv1.") for key in keys), "Missing conv1 weights"
    assert any(key.startswith("layer1.") for key in keys), "Missing layer1 weights"
    assert any(key.startswith("pool.") for key in keys), "Missing GeM weights"
    assert any(key.startswith("bottleneck.") for key in keys), "Missing BNNeck weights"
    print("Checkpoint structure validated")

if __name__ == "__main__":
    main()
'''

script_path = "/kaggle/working/train_09d.py"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)
print(f"Training script written to {script_path}")
print(f"Script size: {os.path.getsize(script_path) / 1024:.1f} KB")

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
DATASETS_ROOT = INPUT_ROOT / "datasets"
WEIGHTS_ROOT = INPUT_ROOT / "mtmc-weights"

def print_tree(root, max_depth=3):
    if not root.exists():
        print(f"  <missing {root}>")
        return
    print(f"  {root}/")
    for path in sorted(root.rglob("*"), key=lambda item: str(item)):
        rel = path.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        indent = "    " * (len(rel.parts) - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}  - {rel}{suffix}")

print("[DEBUG] Recursive listing for /kaggle/input (depth<=3):")
if INPUT_ROOT.exists():
    print_tree(INPUT_ROOT, max_depth=3)
else:
    print("  <missing /kaggle/input>")

if DATASETS_ROOT.exists():
    print("[DEBUG] Recursive listing for /kaggle/input/datasets (depth<=3):")
    print_tree(DATASETS_ROOT, max_depth=3)
else:
    print("[DEBUG] /kaggle/input/datasets does not exist")

print("[DEBUG] Recursive listing for /kaggle/input/mtmc-weights (depth<=3):")
if WEIGHTS_ROOT.exists():
    print_tree(WEIGHTS_ROOT, max_depth=3)
else:
    print("  <missing /kaggle/input/mtmc-weights>")
    print(
        "[WARN] Expected weights dataset mount missing at /kaggle/input/mtmc-weights; "
        "notebook will rely on recursive path resolution."
    )

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "/kaggle/working/train_09d.py"],
    check=False,
    text=True,
)
print(f"Training process exited with code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")